In [ ]:
!pip install rasterio
!pip install rasterstats
%pip install geopandas matplotlib shapely rasterio numpy pandas sklearn-xarray -q
%pip install git+https://github.com/jgrss/geowombat  -q

In [ ]:
import rasterstats

In [ ]:
# Import GeoWombat
import geowombat as gw
# import plotting
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from geowombat.data import l8_224077_20200518_B2
import pandas as pd
import geopandas as gpd
import os
import rasterio
import matplotlib.pyplot as plt
from rasterio.merge import merge
from rasterio.plot import show
from rasterio.mask import mask
from rasterstats import zonal_stats
from rasterstats import zonal_stats
import pyproj
from shapely.ops import transform

Importing shapefiles

In [ ]:
from google.colab import drive
#conectar este notebook sobre todo el drive
drive.mount("/content/drive", force_remount = True)

In [ ]:
departments = gpd.read_file('/content/drive/MyDrive/data/INEI_LIMITE_DEPARTAMENTAL/INEI_LIMITE_DEPARTAMENTAL.shp')

In [ ]:
departments

EPSG to esri 54009

In [ ]:
transformer = pyproj.Transformer.from_crs('epsg:4326', 'esri:54009', always_xy=True)

# Define a function to apply the transformation
def apply_transform(geom):
    return transform(transformer.transform, geom)

# Apply the transformation to the geometries
departments['geometry'] = departments['geometry'].apply(apply_transform)

In [ ]:
tif_files = ['/content/drive/MyDrive/data/rasters/GHS_BUILT_C_MSZ_E2018_GLOBE_R2023A_54009_10_V1_0_R10_C10/GHS_BUILT_C_MSZ_E2018_GLOBE_R2023A_54009_10_V1_0_R10_C10.tif',
             '/content/drive/MyDrive/data/rasters/GHS_BUILT_C_MSZ_E2018_GLOBE_R2023A_54009_10_V1_0_R10_C11/GHS_BUILT_C_MSZ_E2018_GLOBE_R2023A_54009_10_V1_0_R10_C11.tif',
             '/content/drive/MyDrive/data/rasters/GHS_BUILT_C_MSZ_E2018_GLOBE_R2023A_54009_10_V1_0_R10_C12/GHS_BUILT_C_MSZ_E2018_GLOBE_R2023A_54009_10_V1_0_R10_C12.tif',
             '/content/drive/MyDrive/data/rasters/GHS_BUILT_C_MSZ_E2018_GLOBE_R2023A_54009_10_V1_0_R11_C11/GHS_BUILT_C_MSZ_E2018_GLOBE_R2023A_54009_10_V1_0_R11_C11.tif',
             '/content/drive/MyDrive/data/rasters/GHS_BUILT_C_MSZ_E2018_GLOBE_R2023A_54009_10_V1_0_R11_C12/GHS_BUILT_C_MSZ_E2018_GLOBE_R2023A_54009_10_V1_0_R11_C12.tif',
             '/content/drive/MyDrive/data/rasters/GHS_BUILT_C_MSZ_E2018_GLOBE_R2023A_54009_10_V1_0_R12_C11/GHS_BUILT_C_MSZ_E2018_GLOBE_R2023A_54009_10_V1_0_R12_C11.tif',
             '/content/drive/MyDrive/data/rasters/GHS_BUILT_C_MSZ_E2018_GLOBE_R2023A_54009_10_V1_0_R12_C12/GHS_BUILT_C_MSZ_E2018_GLOBE_R2023A_54009_10_V1_0_R12_C12.tif',
             '/content/drive/MyDrive/data/rasters/GHS_BUILT_C_MSZ_E2018_GLOBE_R2023A_54009_10_V1_0_R9_C10/GHS_BUILT_C_MSZ_E2018_GLOBE_R2023A_54009_10_V1_0_R9_C10.tif',
             '/content/drive/MyDrive/data/rasters/GHS_BUILT_C_MSZ_E2018_GLOBE_R2023A_54009_10_V1_0_R9_C11/GHS_BUILT_C_MSZ_E2018_GLOBE_R2023A_54009_10_V1_0_R9_C11.tif',
             '/content/drive/MyDrive/data/rasters/GHS_BUILT_C_MSZ_E2018_GLOBE_R2023A_54009_10_V1_0_R9_C12/GHS_BUILT_C_MSZ_E2018_GLOBE_R2023A_54009_10_V1_0_R9_C12.tif']

In [ ]:
#tif_files = ['/content/drive/MyDrive/data/rasters/GHS_BUILT_C_MSZ_E2018_GLOBE_R2023A_54009_10_V1_0_R10_C10/GHS_BUILT_C_MSZ_E2018_GLOBE_R2023A_54009_10_V1_0_R10_C10.tif']

In [ ]:
dep_stats = []

In [ ]:
#recorrer las imagenes
for raster_path in tif_files:
    stats = zonal_stats(departments, raster_path, stats="count sum", categorical=True, all_touched=True)
    stats_df = pd.DataFrame(stats)
    df = pd.concat([departments.reset_index(drop=True), stats_df], axis=1)
    dep_stats.append(df)

In [ ]:
dep_stats

In [ ]:
# prompt: listar columnas de dep_stats

for df in dep_stats:
    print(df.columns.tolist())


In [ ]:
#unir en un solo dataframe
baseFinal = pd.concat(dep_stats, ignore_index=True)
baseFinal.fillna(0, inplace=True)

In [ ]:
pixel_area = 100
baseFinal['polygon_area'] = baseFinal['geometry'].apply(lambda x: x.area)
for category in range(1, 16):
    category_str = str(category)
    if category_str not in baseFinal.columns:
        baseFinal[category_str] = 0
        baseFinal[f'MSZ_{category}_coverage'] = (baseFinal[category_str] * pixel_area / baseFinal['polygon_area']) * 100

In [ ]:
# Lista de las columnas de cobertura que quieres graficar
coverage_columns = [col for col in baseFinal.columns if 'MSZ_' in col and '_coverage' in col]

# Crear un mapa coroplético para cada categoría de MSZ
for coverage_column in coverage_columns:
    fig, ax = plt.subplots(1, 1, figsize=(10, 6))
    baseFinal.plot(column=coverage_column, ax=ax, legend=True,
                  legend_kwds={'label': f"Coverage of {coverage_column}",
                               'orientation': "horizontal"})
    ax.set_title(f"Map of {coverage_column}")
    plt.show()

In [ ]:
ig, ax = plt.subplots(figsize=(20, 20))

departments.plot(  ax=ax, color='midnightblue', linestyle='dotted',
            edgecolor='white' )
plt.show()

In [ ]:
coverage_columns = [col for col in baseFinal.columns.map(str) if '_coverage' in col]
for coverage_column in coverage_columns:
    fig, ax = plt.subplots(1, 1, figsize=(10, 6))
    baseFinal.plot(column=coverage_column, ax=ax, legend=True,
                  legend_kwds={'label': f"Coverage of {coverage_column}",
                               'orientation': "horizontal"})
    ax.set_title(f"Map of {coverage_column}")
    plt.show()